In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd, numpy as np
from pyspark.sql import functions as F
import formulaic as frm, sklearn as sk, glum as glm
import xgboost as xgb
from sklearn.metrics import d2_tweedie_score
import tempfile, os, mlflow, joblib


In [2]:
def safe_get(var_name, default):
    try:
        return dbutils.widgets.get(var_name)
    except KeyError:
        return default

In [3]:
catalog = safe_get("catalog", "workspace")
schema = safe_get("schema", "mlops_dev")

tbl_ilec_data = (
    spark.read.table(f"{catalog}.{schema}.ilec_data")
    .filter(
        (F.trim(F.col("Insurance_Plan")) == "TERM") &
        (F.col("SOA_Post_Lvl_Ind") != F.lit("PLT")) &
        (F.col("ExpDth_VBT2015wMI_Cnt") > F.lit(0.0))
    )
    .withColumn("DATASET",
        F.when(F.col("Observation_Year") <= F.lit(2017), "TRAIN")
         .otherwise("TEST")
    )
)

tbl_ilec_data.count()

# COMMAND ----------

def agg_data(df : F.DataFrame) -> F.DataFrame:
    return (
        df
        .groupBy(["Observation_Year", "Issue_Year", "Issue_Age", "Sex", "Smoker_Status", "Attained_Age", "Face_Amount_Band"])
        .agg(
            F.sum("Death_Count").alias("Death_Count"),
            F.sum("ExpDth_VBT2015wMI_Cnt").alias("ExpDth_VBT2015wMI_Cnt")
        )
    )

tbl_train = (
    tbl_ilec_data
    .filter(F.col("DATASET") == F.lit("TRAIN"))
)

tbl_test = (
    tbl_ilec_data
    .filter(F.col("DATASET") == F.lit("TEST"))
)

df_train = agg_data(tbl_train).toPandas()
df_test = agg_data(tbl_test).toPandas()


In [4]:
x_mat_formula = frm.Formula(" ~ cr(Attained_Age, df=4, lower_bound=18, upper_bound=90 )*Smoker_Status*Sex + Face_Amount_Band - 1")

In [5]:
X_train = x_mat_formula.get_model_matrix(df_train)
offset_train = np.log(df_train["ExpDth_VBT2015wMI_Cnt"])
y_train = df_train["Death_Count"]

X_val = x_mat_formula.get_model_matrix(df_test)
offset_val = np.log(df_test["ExpDth_VBT2015wMI_Cnt"])
y_val = df_test["Death_Count"]

In [7]:
df_train.attrs = {}
df_train.to_parquet("df_train.parquet")

In [ ]:
coef_table = glmnet.coef_table()

term_groups = [""] * coef_table.shape[0]

def slice_to_indices(s: slice, length: int) -> range:
    return range(*s.indices(length))

term_groups[0] = "intercept"
for term, term_slice in X_train.model_spec.term_slices.items():
    for i in slice_to_indices(term_slice, coef_table.shape[0]):
        term_groups[i + 1] = term
    
term_groups


['intercept',
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
 Smoker_Status,
 Smoker_Status,
 Smoker_Status,
 Sex,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 Face_Amount_Band,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
 cr(Attained_Age,

In [ ]:
(
    pd.DataFrame({
        "term_value" : glmnet.coef_table(),
        "term_group" : term_groups
    })
    .reset_index(drop=False)
    .rename({"index":"term_name"}, axis=1)
).to_csv("sample_mapping.csv", index=False)


In [ ]:
glmnet = glm.GeneralizedLinearRegressor(
    family="poisson",
    alpha_search=True,
    min_alpha_ratio=1e-6
)
glmnet.fit(
    X_train,
    y = y_train,
    offset = offset_train
)

,alpha,None
,l1_ratio,0
,P1,'identity'
,P2,'identity'
,fit_intercept,True
,family,'poisson'
,link,'auto'
,solver,'auto'
,max_iter,100
,max_inner_iter,100000
,gradient_tol,None


In [ ]:
df_train["ExpDth_VBT2015wMI_Cnt"].min()

1.6204893427129718e-07

In [ ]:
from glm_tools import PoissonDecisionTree

dtree = PoissonDecisionTree("Death_Count", "ExpDth_VBT2015wMI_Cnt")
dtree.fit(df_train)
print(dtree)

c:\Users\ande7\Workspace\databricks-ilec\ilec_pipeline\src\explore\glm_tools.py:122: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
c:\Users\ande7\Workspace\databricks-ilec\ilec_pipeline\src\explore\glm_tools.py:122: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)


root: ae=1.0251   y=217815
  - model_pred < 0.837792397: ae=1.2845   y=83877
    - Face_Amount_Band_03: 25,000 - 49,999 < 0.5: ae=1.2083   y=70252
      - Face_Amount_Band_04: 50,000 - 99,999 < 0.5: ae=1.1071   y=52553
      - Face_Amount_Band_04: 50,000 - 99,999 >= 0.5: ae=1.6584   y=17699
    - Face_Amount_Band_03: 25,000 - 49,999 >= 0.5: ae=1.9038   y=13625
      - Issue_Year < 2007.5: ae=1.7455   y=10024
      - Issue_Year >= 2007.5: ae=2.5466   y=3601
  - model_pred >= 0.837792397: ae=0.9100   y=133938
    - Face_Amount_Band_04: 50,000 - 99,999 < 0.5: ae=0.8913   y=126054
      - Face_Amount_Band_05: 100,000 - 249,999 < 0.5: ae=0.8150   y=63659
      - Face_Amount_Band_05: 100,000 - 249,999 >= 0.5: ae=0.9855   y=62395
    - Face_Amount_Band_04: 50,000 - 99,999 >= 0.5: ae=1.3697   y=7884
      - Attained_Age < 68.5: ae=1.5428   y=4371
      - Attained_Age >= 68.5: ae=1.2019   y=3513


In [ ]:
df_train["model_pred"] = glmnet.predict(X_train, offset=offset_train)
dtree2 = PoissonDecisionTree("Death_Count", "model_pred")
dtree2.fit(df_train.drop("ExpDth_VBT2015wMI_Cnt", axis=1))
print(dtree2)

c:\Users\ande7\Workspace\databricks-ilec\ilec_pipeline\src\explore\glm_tools.py:122: RuntimeWarning: divide by zero encountered in log
  term = np.where(y > 0, y * np.log(y / mu), 0.0)
c:\Users\ande7\Workspace\databricks-ilec\ilec_pipeline\src\explore\glm_tools.py:122: RuntimeWarning: invalid value encountered in multiply
  term = np.where(y > 0, y * np.log(y / mu), 0.0)


root: ae=1.0000   y=217815
  - Issue_Year < 2010.5: ae=0.9793   y=190264
    - Issue_Year < 2006.5: ae=0.9665   y=151140
      - Issue_Year < 1996.5: ae=1.0148   y=42569
      - Issue_Year >= 1996.5: ae=0.9489   y=108571
    - Issue_Year >= 2006.5: ae=1.0319   y=39124
      - Issue_Age < 74.5: ae=1.0270   y=38397
      - Issue_Age >= 74.5: ae=1.3800   y=727
  - Issue_Year >= 2010.5: ae=1.1710   y=27551
    - Issue_Age < 74.5: ae=1.1470   y=26464
      - Face_Amount_Band_04: 50,000 - 99,999 < 0.5: ae=1.1122   y=23321
      - Face_Amount_Band_04: 50,000 - 99,999 >= 0.5: ae=1.4930   y=3143
    - Issue_Age >= 74.5: ae=2.3872   y=1087
      - Face_Amount_Band_05: 100,000 - 249,999 < 0.5: ae=2.6368   y=983
      - Face_Amount_Band_05: 100,000 - 249,999 >= 0.5: ae=1.2600   y=104


In [1]:
pd.DataFrame(df_train).to_parquet("test.parquet")

NameError: name 'pd' is not defined

In [20]:
X_train.model_spec.factor_variables

{Sex: {'Sex'},
 Smoker_Status: {'Smoker_Status'},
 Face_Amount_Band: {'Face_Amount_Band'},
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90): {'Attained_Age',
  'cr'}}

In [19]:
X_train.model_spec.factor_terms

{cr(Attained_Age, df=4, lower_bound=18, upper_bound=90): {cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 Smoker_Status: {Smoker_Status,
  Smoker_Status:Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 Sex: {Sex,
  Smoker_Status:Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 Face_Amount_Band: {Face_Amount_Band}}

In [22]:
X_train.model_spec.variable_terms

{'cr': {cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 'Attained_Age': {cr(Attained_Age, df=4, lower_bound=18, upper_bound=90),
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 'Smoker_Status': {Smoker_Status,
  Smoker_Status:Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 'Sex': {Sex,
  Smoker_Status:Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex},
 'Face_Amount_Band': {Face_Amount_Band}}

In [23]:
X_train.model_spec.variables_by_source

{'data': {'Attained_Age', 'Face_Amount_Band', 'Sex', 'Smoker_Status'},
 'transforms': {'cr'}}

In [27]:
X_train.model_spec.term_factors

{cr(Attained_Age, df=4, lower_bound=18, upper_bound=90): {cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)},
 Smoker_Status: {Smoker_Status},
 Sex: {Sex},
 Face_Amount_Band: {Face_Amount_Band},
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status: {Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)},
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex: {Sex,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)},
 Smoker_Status:Sex: {Sex, Smoker_Status},
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex: {Sex,
  Smoker_Status,
  cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)}}

In [31]:
X_train.model_spec.column_names

('cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[1]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[2]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[3]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[4]',
 'Smoker_Status[NS]',
 'Smoker_Status[S]',
 'Smoker_Status[U]',
 'Sex[T.M]',
 'Face_Amount_Band[T.02: 10,000 - 24,999]',
 'Face_Amount_Band[T.03: 25,000 - 49,999]',
 'Face_Amount_Band[T.04: 50,000 - 99,999]',
 'Face_Amount_Band[T.05: 100,000 - 249,999]',
 'Face_Amount_Band[T.06: 250,000 - 499,999]',
 'Face_Amount_Band[T.07: 500,000 - 999,999]',
 'Face_Amount_Band[T.08: 1,000,000 - 2,499,999]',
 'Face_Amount_Band[T.09: 2,500,000 - 4,999,999]',
 'Face_Amount_Band[T.10: 5,000,000 - 9,999,999]',
 'Face_Amount_Band[T.11: 10,000,000+]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[1]:Smoker_Status[T.S]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[2]:Smoker_Status[T.S]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=

In [33]:
X_train.model_spec.term_slices["Sex"].

slice(7, 8, None)

In [35]:
glmnet.coef_table()

intercept                                                                                0.293820
cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[1]                                0.201978
cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[2]                                0.036141
cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[3]                               -0.133154
cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[4]                               -0.104965
Smoker_Status[NS]                                                                        0.074542
Smoker_Status[S]                                                                         0.052475
Smoker_Status[U]                                                                        -0.127016
Sex[T.M]                                                                                 0.065321
Face_Amount_Band[T.02: 10,000 - 24,999]                                                  0.426490
Face_Amount_Band[T.0

In [32]:
X_train.model_spec.term_slices

{cr(Attained_Age, df=4, lower_bound=18, upper_bound=90): slice(0, 4, None),
 Smoker_Status: slice(4, 7, None),
 Sex: slice(7, 8, None),
 Face_Amount_Band: slice(8, 18, None),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status: slice(18, 26, None),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Sex: slice(26, 30, None),
 Smoker_Status:Sex: slice(30, 32, None),
 cr(Attained_Age, df=4, lower_bound=18, upper_bound=90):Smoker_Status:Sex: slice(32, 40, None)}

In [24]:
X_train.model_spec.column_names

('cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[1]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[2]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[3]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[4]',
 'Smoker_Status[NS]',
 'Smoker_Status[S]',
 'Smoker_Status[U]',
 'Sex[T.M]',
 'Face_Amount_Band[T.02: 10,000 - 24,999]',
 'Face_Amount_Band[T.03: 25,000 - 49,999]',
 'Face_Amount_Band[T.04: 50,000 - 99,999]',
 'Face_Amount_Band[T.05: 100,000 - 249,999]',
 'Face_Amount_Band[T.06: 250,000 - 499,999]',
 'Face_Amount_Band[T.07: 500,000 - 999,999]',
 'Face_Amount_Band[T.08: 1,000,000 - 2,499,999]',
 'Face_Amount_Band[T.09: 2,500,000 - 4,999,999]',
 'Face_Amount_Band[T.10: 5,000,000 - 9,999,999]',
 'Face_Amount_Band[T.11: 10,000,000+]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[1]:Smoker_Status[T.S]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=90)[2]:Smoker_Status[T.S]',
 'cr(Attained_Age, df=4, lower_bound=18, upper_bound=

In [ ]:
str_cols = list(map(
    lambda c: str(c),
    df_train.select_dtypes(
    include=['object', 'string']).columns
))

response_cols = [
    "Death_Count",
    "ExpDth_VBT2015wMI_Cnt"
]

num_cols = list(set(df_train.columns).difference(set(str_cols)).difference(set(response_cols)))

xfrm = [
    (
        "ohe",
        sk.preprocessing.OneHotEncoder(
            drop="first", handle_unknown="ignore",
        ),
        str_cols
    ),
    (
        "num", "passthrough", num_cols
    )
]

preproc = sk.compose.ColumnTransformer(
    transformers=xfrm,
    remainder="drop",
    verbose_feature_names_out=True
)
    
xgb_train = preproc.fit_transform(df_train)
xgb_offset = np.log(glmnet.predict(X_train, offset=offset_train))

In [ ]:
df_train["Death_Count"].to_numpy('float32')

array([0., 0., 1., ..., 0., 0., 0.], dtype=float32)

In [ ]:
df_xgb_train = pd.DataFrame(preproc.fit_transform(df_train))
df_xgb_train.columns = preproc.get_feature_names_out().tolist()

In [ ]:
xgb_mat = xgb.DMatrix(
    data=xgb_train,
    label=y_train,
    base_margin=xgb_offset, feature_names=preproc.get_feature_names_out().tolist()
  
)

params = {
    'objective': 'count:poisson',
    'max_depth': 3,
    'tree_method': 'exact',
    'grow_policy': 'lossguide',
    'seed':0
}

bst = xgb.train(
    params=params,
    dtrain=xgb_mat,
    num_boost_round=1
)


In [ ]:
import json
from glm_tools import PoissonDecisionTree
# -------------------------------------------------
# 2) Parse tree JSON
# -------------------------------------------------
tree_json = json.loads(bst.get_dump(dump_format="json")[0])

# -------------------------------------------------
# 4) Compute split-level deviance table
# -------------------------------------------------
split_stats = collect_split_stats_pretty_path(tree_json, df_xgb_train, y_train, np.exp(xgb_offset))
split_df = pd.DataFrame(split_stats)

pretty_print_split_stats(split_df)

# print(split_df[[
#     "nodeid", "feature", "threshold",
#     "dev_parent", "dev_children", "dev_reduction",
#     "xgb_gain"
# ]])

POISSON SPLIT SUMMARY
Node 0
  Path: ROOT
  Split: num__Issue_Year < 2,010.5000
  Parent
    n=896932  actual=217815  exp=217,815.2402  ae=1.0000  dev=335,768.1266
  Left child
    path: (num__Issue_Year < 2010.5 OR num__Issue_Year missing)
    n=759806  actual=190264  exp=194,286.6069  ae=0.9793  dev=282,089.1103
  Right child
    path: num__Issue_Year >= 2010.5
    n=137126  actual=27551  exp=23,528.6333  ae=1.1710  dev=52,943.6458
  Improvement
    children_dev=335,032.7561  dev_reduction=735.3705  xgb_gain=NA
------------------------------------------------------------------------------------------------------------------------
Node 1
  Path: (num__Issue_Year < 2010.5 OR num__Issue_Year missing)
  Split: num__Issue_Year < 2,006.5000
  Parent
    n=759806  actual=190264  exp=194,286.6069  ae=0.9793  dev=282,089.1103
  Left child
    path: (num__Issue_Year < 2010.5 OR num__Issue_Year missing) AND (num__Issue_Year < 2006.5 OR num__Issue_Year missing)
    n=630664  actual=151140  exp=1